## Components in LlamaIndex

In [1]:
%pip install llama-index datasets llama-index-callbacks-arize-phoenix arize-phoenix llama-index-vector-stores-chroma llama-index-llms-huggingface-api llama-index-embeddings-huggingface -U -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\sobha\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
from huggingface_hub import login

login()

### Create a QueryEngine for retrieval augmented generation

Setting up the persona database
We will be using personas from the dvilasuero/finepersonas-v0.1-tiny dataset. This dataset contains 5K personas that will be attending the party!

Let's load the dataset and store it as files in the data directory

In [3]:
from datasets import load_dataset
from pathlib import Path

dataset = load_dataset(path="dvilasuero/finepersonas-v0.1-tiny", split="train")

Path("data").mkdir(parents=True, exist_ok=True)
for i, persona in enumerate(dataset):
    with open(Path("data") / f"persona_{i}.txt", "w", encoding="utf-8") as f:
        f.write(persona["persona"])

C:\Users\sobha\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Awesome, now we have a local directory with all the personas that will be attending the party, we can load and index!

#### Loading and embedding persona documents
We will use the SimpleDirectoryReader to load the persona descriptions from the data directory. This will return a list of Document objects.

In [4]:
from llama_index.core import SimpleDirectoryReader

reader = SimpleDirectoryReader(input_dir="data")
documents = reader.load_data()
len(documents)

5000

Now we have a list of Document objects, we can use the IngestionPipeline to create nodes from the documents and prepare them for the QueryEngine. We will use the SentenceSplitter to split the documents into smaller chunks and the HuggingFaceEmbedding to embed the chunks.

In [5]:
%pip install -U transformers sentence-transformers huggingface_hub

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\sobha\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
import os

from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline

# the Microsoft Store Python breaks llama-index's default model cache (0-byte files), so use the regular HF cache
HF_CACHE = os.path.expanduser("~/.cache/huggingface/hub")

# create the pipeline with transformations
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(),
        HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5", cache_folder=HF_CACHE),
    ]
)

# run the pipeline sync or async
nodes = await pipeline.arun(documents=documents[:10])
nodes

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1518.14it/s]


[TextNode(id_='e749e05e-6a91-48c1-b927-dc42a6756533', embedding=[-0.04602069780230522, -0.017991214990615845, 0.03904610872268677, 0.006559292785823345, 0.008312386460602283, -0.03702801465988159, -0.01872582733631134, 0.021880609914660454, -0.030854521319270134, -0.04769223555922508, 0.025162188336253166, -0.025820570066571236, 0.02204062044620514, 0.048974018543958664, 0.01802593655884266, 0.013407045975327492, -0.014733856543898582, 0.061176598072052, 0.006909106858074665, 0.0007446932722814381, 0.022678477689623833, -0.03953009471297264, 0.06247144564986229, -0.0291094072163105, -0.05376178026199341, 0.04422307759523392, 0.00976734608411789, -0.04036363214254379, -0.00021618731261696666, -0.1501312106847763, -0.02130799926817417, -0.00818542018532753, 0.02332705818116665, 0.006675187032669783, 0.054803043603897095, 0.022929245606064796, -0.010637128725647926, 0.02729913406074047, -0.01714146137237549, 0.023837842047214508, 0.024997202679514885, 0.0016260871198028326, -0.00625715637

As, you can see, we have created a list of Node objects, which are just chunks of text from the original documents. Let's explore how we can add these nodes to a vector store.

### Storing and indexing documents

Since we are using an ingestion pipeline, we can directly attach a vector store to the pipeline to populate it. In this case, we will use Chroma to store our documents. Let's run the pipeline again with the vector store attached. The IngestionPipeline caches the operations so this should be fast!

In [7]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore

db = chromadb.PersistentClient(path="./alfred_chroma_db")
chroma_collection = db.get_or_create_collection(name="alfred")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(),
        HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5", cache_folder=HF_CACHE),
    ],
    vector_store=vector_store,
)

nodes = await pipeline.arun(documents=documents[:10])
len(nodes)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2756.35it/s]


10

In [8]:
%pip list 2>$null | findstr /i "transformers sentence-transformers huggingface"


huggingface_hub                           2.3.0
langchain-huggingface                     1.2.2
llama-index-embeddings-huggingface        0.8.0
llama-index-llms-huggingface-api          0.8.0
sentence-transformers                     6.2.0
transformers                              5.21.0
Note: you may need to restart the kernel to use updated packages.


#### Storing and indexing documents
Since we are using an ingestion pipeline, we can directly attach a vector store to the pipeline to populate it. In this case, we will use Chroma to store our documents. Let's run the pipeline again with the vector store attached. The IngestionPipeline caches the operations so this should be fast!

In [10]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore

db = chromadb.PersistentClient(path="./alfred_chroma_db")
chroma_collection = db.get_or_create_collection(name="alfred")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(),
        HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5", cache_folder=HF_CACHE),
    ],
    vector_store=vector_store,
)

nodes = await pipeline.arun(documents=documents[:10])
len(nodes)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3616.75it/s]


10

We can create a VectorStoreIndex from the vector store and use it to query the documents by passing the vector store and embedding model to the from_vector_store() method.

In [12]:
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5", cache_folder=HF_CACHE)
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store, embed_model=embed_model
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2257.98it/s]


We don't need to worry about persisting the index to disk, as it is automatically saved within the ChromaVectorStore object and the passed directory path.

#### Querying the index
Now that we have our index, we can use it to query the documents. Let's create a QueryEngine from the index and use it to query the documents using a specific response mode.

In [13]:
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
import nest_asyncio

nest_asyncio.apply()  # This is needed to run the query engine
llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")
query_engine = index.as_query_engine(
    llm=llm,
    response_mode="tree_summarize",
)
response = query_engine.query(
    "Respond using a persona that describes author and travel experiences?"
)
response

Response(response="This individual is deeply engaged in the study of Cypriot culture, history, and society. They have dedicated significant time to research and reside in Cyprus, immersing themselves in the local customs and way of life to develop a profound understanding of the island's people and their traditions.", source_nodes=[NodeWithScore(node=TextNode(id_='b8bf430c-d769-469b-86ef-c13f0f09716f', embedding=None, metadata={'file_path': 'd:\\Projects\\Personal Projects\\learn-AI\\3-ai-agents-course\\unit-2-ai-agents-framework\\unit-2.2-llamaindex-framework\\project-1-intro-to-llamahub\\data\\persona_1.txt', 'file_name': 'persona_1.txt', 'file_type': 'text/plain', 'file_size': 266, 'creation_date': '2026-10-26', 'last_modified_date': '2026-10-27'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', '

#### Evaluation and observability
LlamaIndex provides built-in evaluation tools to assess response quality. These evaluators leverage LLMs to analyze responses across different dimensions. We can now check if the query is faithful to the original persona.

In [14]:
from llama_index.core.evaluation import FaithfulnessEvaluator

# query index
evaluator = FaithfulnessEvaluator(llm=llm)
eval_result = evaluator.evaluate_response(response=response)
eval_result.passing

True